# demo_shapelet_mcmc.jl — Notebook Edition

**End-to-end MCMC**: SIS lens + shapelet source reconstruction.

- Mock data generation (shapelet-generated truth)
- `build_design` / `solve_coeffs` inside MCMC logp
- `lens_mh_multistart` chain + diagnostics
- Posterior and residual visualisation

> Key insight: MCMC only samples 4 parameters (`theta_E`, `beta`, `xc_src`, `yc_src`). The 28 shapelet coefficients are solved analytically at each step — a technique called *analytical marginalization*.

---
## Block 0: Imports & Seed

In [ ]:
using Jens, Cosmology, Statistics, LinearAlgebra, Random, Printf, Plots
using Jens.LensShapelet: ShapeletBasis, n_basis, build_design,
    solve_coeffs, shapelet_model, build_reg_matrix
using Jens.LensMH: lens_mh_multistart, lens_mh, chain, chain_stats

Random.seed!(42)

---
## Block 1: Mock Data Generation

### 1a — Cosmology + Grid + Truth Parameters

Create a Flat ΛCDM cosmology, a 64×64 pixel grid, and define truth values for all parameters. `NMAX_TRUE=6` means 28 shapelet basis functions: (6+1)×(6+2)/2 = 28.

In [ ]:
println("=== Generating mock data (shapelet truth) ===\n")

cosmo = Cosmology.FlatLCDM(0.7, 0.3, 0.0, 0.0)
grid  = Jens.LensGenerator.GenGrid(pix_n=64)   # 64² = 4096 pix; masked ~1000

# Truth parameters
THETA_E_TRUE = 0.8
BETA_TRUE    = 0.10
SRC_X_TRUE   = 0.03
SRC_Y_TRUE   = 0.02
NMAX_TRUE    = 6                                # 28 basis functions
SIGMA_NOISE  = 0.003

n_basis_true = (NMAX_TRUE + 1) * (NMAX_TRUE + 2) ÷ 2

@show THETA_E_TRUE BETA_TRUE NMAX_TRUE n_basis_true SIGMA_NOISE

### 1b — Truth Lens + ForwardModel

Build an SIS lens (singular isothermal sphere) centered at origin. The `ForwardModel` binds the lens plane (z=0.3), source plane (z=1.5), and pixel grid together. The `GaussianSphere` in the source plane is just a placeholder — the real source is rendered via shapelet coefficients.

In [ ]:
lens_truth = Jens.LensBase.SingleModel(
    Jens.LensModel.SIS.SIS;
    theta_E=THETA_E_TRUE, xcentre=0.0, ycentre=0.0)

sys_truth = Jens.LensSystem.ForwardModel(;
    lens_plane   = Jens.LensGenerator.LensedPlane(
        lens_truth; z_lens=0.3, cosmology=cosmo),
    source_plane = Jens.LightModel.ExtendedSource(
        Jens.LightModel.GaussianLight.GaussianSphere;
        amp=1.0, sigma=0.01),
    grid         = grid,
    z_source     = 1.5,
)

println("Lens + ForwardModel created.")

### 1c — Mask

Circular mask (radius 2 arcsec). Only pixels inside the mask participate in fitting — reduces the problem from 4096 pixels to ~1000.

In [ ]:
mask = Jens.LensMask.circular_mask(grid, 2.0)
mask_idx = findall(vec(mask))
n_pix = length(mask_idx)

@show n_pix

### 1d — Build Design Matrix & Generate Truth Coefficients

`build_design` is the workhorse: for each of the 28 shapelet basis functions, it ray-traces through the lens model to produce the corresponding lensed image on the pixel grid. The result is $A$, a $n_\text{pix} \times n_\text{basis}$ matrix where each column is one basis function's lensed image.

We create two versions:
- `A_masked_true`: rows = masked pixels only (~1000 × 28) — for fitting
- `A_conv_true`: rows = all pixels (4096 × 28) — for rendering

In [ ]:
basis_true = ShapeletBasis(NMAX_TRUE, BETA_TRUE)

tmp_data = zeros(size(grid.xg))  # placeholder — needed for API but unused

A_masked_true, A_conv_true, _ = build_design(
    sys_truth, basis_true, tmp_data, mask;
    xc_src=SRC_X_TRUE, yc_src=SRC_Y_TRUE)

@show size(A_masked_true) size(A_conv_true)

### 1e — Set Shapelet Coefficients (Truth)

The image is $I = A \cdot c$. We manually set a few low-order coefficients and leave the rest as small random values.

In [ ]:
c_truth = randn(n_basis_true) .* 0.3
c_truth[1] = 1.0                                  # DC component (total flux)
c_truth[2] = 0.3                                  # n1=1,n2=0 → dipole x
c_truth[3] = -0.2                                 # n1=0,n2=1 → dipole y

println("First 5 coefficients: ", round.(c_truth[1:5], digits=3))

### 1f — Render Truth Image + Add Noise

Normalize so peak flux ≈ 1, then add Gaussian noise.

In [ ]:
data_true = reshape(A_conv_true * c_truth, size(grid.xg))
data_true_max = maximum(abs, data_true)
data_true ./= data_true_max                        # normalise to ~1

# Rescale coefficients to match
c_truth ./= data_true_max
data_true = reshape(A_conv_true * c_truth, size(grid.xg))

# Add noise
data = data_true .+ SIGMA_NOISE .* randn(size(data_true))
snr = maximum(data[mask]) / SIGMA_NOISE

println("  θ_E truth  = $THETA_E_TRUE")
println("  β truth    = $BETA_TRUE")
println("  src centre = ($SRC_X_TRUE, $SRC_Y_TRUE)")
println("  noise σ    = $SIGMA_NOISE")
println("  peak S/N   = $(round(snr, digits=1))")
println("  masked pix = $n_pix")
println("  n_basis    = $n_basis_true")

---
## Block 2: MCMC Setup

We use the same `n_max` as truth (well-specified model). MCMC only samples 4 parameters: `[theta_E, beta, xc_src, yc_src]`. The 28 shapelet coefficients are marginalized analytically via least-squares.

In [ ]:
SHAPELET_NMAX_MCMC = NMAX_TRUE    # must = truth for well-specified model
N_MCMC = (SHAPELET_NMAX_MCMC + 1) * (SHAPELET_NMAX_MCMC + 2) ÷ 2

println("=== MCMC setup ===")
println("  MCMC n_max  = $SHAPELET_NMAX_MCMC ($N_MCMC coefficients)")
println("  Design size = $n_pix × $N_MCMC")

---
## Block 3: Log-Probability Function

This is the heart of the inference. At each MCMC step:

1. Build SIS lens with current `theta_E`
2. Build `ForwardModel` → ray-trace shapelet basis → design matrix $A$
3. Solve $c = (A^T A + \lambda I)^{-1} A^T d$ (linear least-squares with Tikhonov regularization)
4. Compute $\chi^2 = \sum (d - Ac)^2 / \sigma^2$
5. Return $\log P = -\chi^2/2$

`Base.invokelatest` is a Julia 1.12 workaround for world-age issues with dynamically-compiled closures.

In [ ]:
function safe_shapelet_logp(params)
    return Base.invokelatest(params) do p
        theta_E = p[1]
        beta    = p[2]
        xc_src  = p[3]
        yc_src  = p[4]

        # Hard bounds (uniform prior)
        if theta_E < 0.1 || theta_E > 2.5 ||
           beta < 0.01 || beta > 0.50 ||
           abs(xc_src) > 0.3 || abs(yc_src) > 0.3
            return -Inf
        end

        lens = Jens.LensBase.SingleModel(
            Jens.LensModel.SIS.SIS;
            theta_E=theta_E, xcentre=0.0, ycentre=0.0)

        sys = Jens.LensSystem.ForwardModel(;
            lens_plane   = Jens.LensGenerator.LensedPlane(
                lens; z_lens=0.3, cosmology=cosmo),
            source_plane = Jens.LightModel.ExtendedSource(
                Jens.LightModel.GaussianLight.GaussianSphere;
                amp=1.0, sigma=0.01),
            grid         = grid,
            z_source     = 1.5,
        )

        b = ShapeletBasis(SHAPELET_NMAX_MCMC, beta)

        A_masked, _, _ = build_design(sys, b, data, mask;
                                       xc_src=xc_src, yc_src=yc_src)

        _, chi2 = solve_coeffs(A_masked, data, mask_idx, SIGMA_NOISE;
                                lambda=0.01)

        return -chi2 / 2
    end
end

println("logp function defined.")

---
## Block 4: Run MCMC

`lens_mh_multistart` launches 24 independent Metropolis-Hastings chains from random starting points in the hypercube defined by `lower` and `upper`. Each chain:
- Warmup: 1500 steps (adaptive step-size tuning)
- Production: 2000 steps
- Total: 24 × 3500 = 84,000 `safe_shapelet_logp` calls

In [ ]:
println("=== Running MCMC ===\n")

lower = [0.3,   0.03,    -0.15,  -0.15]
upper = [2.0,   0.30,     0.15,   0.15]
#         θ_E    β       xc_src   yc_src

n_params = length(lower)

@time result = lens_mh_multistart(safe_shapelet_logp, lower, upper;
                                   n_starts=24, n_warmup=1500, n=2000, seed=42)

samples = chain(result; burn=500)
means, stds = chain_stats(result; burn=500)

acc_rate = result.accepted / 2000 * 100
println("  Acceptance rate: $(round(acc_rate, digits=1))%")

### 4b — Compare Posterior to Truth

For each parameter, show: truth value, posterior mean ± std, and the deviation in units of σ.

In [ ]:
println("\n─── MCMC Results ───")
labels = ["θ_E", "β", "xc_src", "yc_src"]
truths = [THETA_E_TRUE, BETA_TRUE, SRC_X_TRUE, SRC_Y_TRUE]
println(rpad("  Parameter", 12), rpad("Truth", 10), rpad("Posterior", 24), "Δ")
for i in 1:n_params
    delta = means[i] - truths[i]
    sigma_delta = delta / stds[i]
    println(@sprintf("  %-10s  %6.3f   %6.3f ± %6.3f   %+.3f  (%+.1fσ)",
                     labels[i], truths[i], means[i], stds[i], delta, sigma_delta))
end

---
## Block 5: Reconstruct Best-Fit Model

Use the posterior means as best-fit parameters, build the system one last time, solve for shapelet coefficients, and compute residuals.

In [ ]:
println("\n=== Reconstructing best-fit model ===\n")

best_theta_E = means[1]
best_beta    = means[2]
best_xc      = means[3]
best_yc      = means[4]

lens_best = Jens.LensBase.SingleModel(
    Jens.LensModel.SIS.SIS;
    theta_E=best_theta_E, xcentre=0.0, ycentre=0.0)

sys_best = Jens.LensSystem.ForwardModel(;
    lens_plane   = Jens.LensGenerator.LensedPlane(
        lens_best; z_lens=0.3, cosmology=cosmo),
    source_plane = Jens.LightModel.ExtendedSource(
        Jens.LightModel.GaussianLight.GaussianSphere;
        amp=1.0, sigma=0.01),
    grid         = grid,
    z_source     = 1.5,
)

basis_best = ShapeletBasis(SHAPELET_NMAX_MCMC, best_beta)
A_masked_best, A_conv_best, _ = build_design(
    sys_best, basis_best, data, mask;
    xc_src=best_xc, yc_src=best_yc)
c_best, chi2_best = solve_coeffs(A_masked_best, data, mask_idx, SIGMA_NOISE;
                                  lambda=0.01)
model_best = shapelet_model(A_conv_best, c_best, size(grid.xg, 1), size(grid.xg, 2))

residual = (data .- model_best) .* mask
chi2_red = chi2_best / (n_pix - N_MCMC)
max_res = maximum(abs, residual[mask])

println("  reduced χ² = ", round(chi2_red, digits=3))
println("  max residual = ", round(max_res, digits=5),
        "  ($(round(max_res/SIGMA_NOISE, digits=1))σ)")

---
## Block 6: Visualization

Three-panel plot: **Mock Data | Best-Fit Model | Residual**

- Data & model: `magma` colormap, same color scale
- Residual: `RdBu` (red-white-blue), clipped at ±4σ

In [ ]:
println("=== Plotting ===\n")

xvec = grid.xg[:, 1]
yvec = grid.yg[1, :]
clim_data = (-0.05, maximum(data[mask]) * 1.1)

p1 = heatmap(xvec, yvec, data';
    aspect_ratio=:equal, title="Mock (S/N≈$(round(snr,digits=1)))",
    c=:magma, xlabel="x [arcsec]", ylabel="y [arcsec]",
    clim=clim_data, colorbar=true)

p2 = heatmap(xvec, yvec, model_best';
    aspect_ratio=:equal, title="Shapelet (n_max=$NMAX_TRUE)",
    c=:magma, xlabel="x [arcsec]", ylabel="y [arcsec]",
    clim=clim_data, colorbar=true)

p3 = heatmap(xvec, yvec, residual';
    aspect_ratio=:equal, title="Residual (σ=$SIGMA_NOISE)",
    c=:RdBu, xlabel="x [arcsec]", ylabel="y [arcsec]",
    clim=(-4*SIGMA_NOISE, 4*SIGMA_NOISE), colorbar=true)

fig = plot(p1, p2, p3; layout=(1, 3), size=(1500, 460),
    plot_title="Shapelet MCMC  |  " *
    "θ_E=$(round(best_theta_E,digits=3))  " *
    "β=$(round(best_beta,digits=3))  " *
    "χ²_ν=$(round(chi2_red,digits=2))")

display(fig)

---
## Summary

| Aspect | Detail |
|--------|--------|
| Lens model | SIS (1 free param: `theta_E`) |
| Source model | Shapelets, `n_max=6` (28 basis functions) |
| MCMC parameters | 4: `[theta_E, beta, xc_src, yc_src]` |
| Marginalized | 28 shapelet coefficients (via least-squares) |
| Sampler | Multi-start Metropolis-Hastings (24 chains) |
| Key trick | Coefficients analytically marginalized → MCMC only needs 4 params |

### Data Flow
```
Truth params → Design matrix A → A×c → Truth image → +noise → Mock data
                                               ↓
MCMC [θ_E,β,xc,yc] ← each step: build A→solve c→χ²→logp
                                               ↓
Posterior mean → Best-fit reconstruction → Residual → Plot
```